In [1]:
##Block 1 - Control
##Block 2 - Delayed Temperature Increase Experimental
##Block 3 - Altered Pathogenic Ratios Experimental
##Block 4 - Standard Experiment

In [ ]:
CONTROL_MODE = True  # True = A and B identical, False = original experiment

import math
import random
import csv
import os
from dataclasses import dataclass
from typing import Dict, Set


# =========================
# CONFIG
# =========================

SEED = None

N_TURNS = 250
N_PLANT_SPECIES = 2000
N_MICROBE_STRAINS = 5000

VARIABLES = ["sunlight", "water", "carbon_dioxide", "nutrients", "temperature"]
FINITE_RESOURCE_BUCKETS = ["water", "carbon_dioxide", "nutrients"]

RESOURCE_CAPACITY = 50_000
RESOURCE_REPLENISH_FRACTION = 0.10
CARRYING_CAPACITY = 5_000

TEMPERATURE_START = 0.5
TEMPERATURE_MAX = 2.5
TEMPERATURE_INCREMENT = 0.001

PLANT_INITIAL_SPECIMENS = 20
MICROBE_INITIAL_COLONIES = 20

MICROBE_TYPE_WEIGHTS = {
    "neutral": 0.85,
    "beneficial": 0.12,
    "pathogenic": 0.03,
}

BENEFICIAL_EFFECT_MIN = 0.01
BENEFICIAL_EFFECT_MAX = 0.10
PATHOGENIC_EFFECT_MIN = -0.50
PATHOGENIC_EFFECT_MAX = -0.10

MICROBE_TARGET_MIN = 2
MICROBE_TARGET_MAX = 50

PLASMID_TEMP_BONUS = 0.15

HGT_FLAT_RATE_MIN = 0.001
HGT_FLAT_RATE_MAX = 0.1


# =========================
# HELPERS
# =========================

def clamp(x, lo, hi):
    return max(lo, min(hi, x))


def weighted_choice(d):
    r = random.random()
    s = 0
    for k, v in d.items():
        s += v
        if r <= s:
            return k
    return list(d.keys())[-1]


@dataclass
class Range:
    low: float
    high: float

    @property
    def midpoint(self):
        return (self.low + self.high) / 2

    @property
    def half_range(self):
        return (self.high - self.low) / 2

    def contains(self, x):
        return self.low <= x <= self.high

    def modified(self, effect):
        mid = self.midpoint
        hr = self.half_range * (1 + effect)
        return Range(
            clamp(mid - hr, 0.01, 1.0),
            clamp(mid + hr, 0.01, 1.0)
        )


@dataclass
class Bucket:
    name: str
    rating: float
    capacity: float
    amount: float
    growth: float

    def replenish(self):
        if self.name in FINITE_RESOURCE_BUCKETS:
            self.amount = min(self.capacity, self.amount + self.growth)


@dataclass
class Plant:
    name: str
    ranges: Dict[str, Range]
    specimens: int
    alive: bool = True


@dataclass
class Microbe:
    name: str
    ranges: Dict[str, Range]
    effect_type: str
    effect_size: float
    affects: Set[str]
    colonies: int
    alive: bool = True
    plasmid_colonies: int = 0

    def effective_ranges(self):
        rr = {k: Range(v.low, v.high) for k, v in self.ranges.items()}
        if self.colonies > 0 and self.plasmid_colonies > 0:
            t = rr["temperature"]
            rr["temperature"] = Range(t.low, clamp(t.high + PLASMID_TEMP_BONUS, 0.01, 1.0))
        return rr


# =========================
# CSV LOGGER
# =========================

def append_summary_to_csv(plants_a, microbes_a, plants_b, microbes_b, hgt_total, seed, filename="run_summary_control.csv"):
    a_plants = sum(1 for p in plants_a if p.alive and p.specimens > 0)
    a_microbes = sum(1 for m in microbes_a if m.alive and m.colonies > 0)

    b_plants = sum(1 for p in plants_b if p.alive and p.specimens > 0)
    b_microbes = sum(1 for m in microbes_b if m.alive and m.colonies > 0)

    file_exists = os.path.isfile(filename)

    with open(filename, "a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow([
                "seed",
                "A_plants", "A_microbes",
                "A_beneficial", "A_neutral", "A_pathogenic",
                "B_plants", "B_microbes",
                "B_beneficial", "B_neutral", "B_pathogenic",
                "HGT"
                ])
        a_counts = count_microbe_types(microbes_a)
        b_counts = count_microbe_types(microbes_b)

        writer.writerow([
            seed,
            a_plants, a_microbes,
            a_counts["beneficial"], a_counts["neutral"], a_counts["pathogenic"],
            b_plants, b_microbes,
            b_counts["beneficial"], b_counts["neutral"], b_counts["pathogenic"],
            hgt_total
        ])

# =========================
# GENERATION
# =========================

def random_range():
    return Range(random.uniform(0.01, 0.5), random.uniform(0.51, 1.0))


def make_environment():
    buckets = {}
    for v in VARIABLES:
        if v == "temperature":
            buckets[v] = Bucket(v, TEMPERATURE_START, float("inf"), float("inf"), 0)
        elif v == "sunlight":
            buckets[v] = Bucket(v, random.uniform(0.01, 1.0), float("inf"), float("inf"), 0)
        else:
            buckets[v] = Bucket(v, random.uniform(0.01, 1.0), RESOURCE_CAPACITY, RESOURCE_CAPACITY, RESOURCE_CAPACITY * 0.1)
    return buckets


def make_plants(n):
    return [Plant(f"P{i}", {v: random_range() for v in VARIABLES}, PLANT_INITIAL_SPECIMENS) for i in range(n)]


def make_microbes(n, plant_names):
    microbes = []
    for i in range(n):
        et = weighted_choice(MICROBE_TYPE_WEIGHTS)
        if et == "neutral":
            es = 0
        elif et == "beneficial":
            es = random.uniform(BENEFICIAL_EFFECT_MIN, BENEFICIAL_EFFECT_MAX)
        else:
            es = random.uniform(PATHOGENIC_EFFECT_MIN, PATHOGENIC_EFFECT_MAX)

        k = random.randint(MICROBE_TARGET_MIN, min(MICROBE_TARGET_MAX, len(plant_names)))

        microbes.append(
            Microbe(
                f"M{i}",
                {v: random_range() for v in VARIABLES},
                et,
                es,
                set(random.sample(plant_names, k)),
                MICROBE_INITIAL_COLONIES
            )
        )
    return microbes


def make_modified_microbe(env, plant_names):
    rr = {}
    for v in VARIABLES:
        val = env[v].rating
        rr[v] = Range(clamp(val - 0.25, 0.01, 1.0), clamp(val + 0.25, 0.01, 1.0))

    k = random.randint(MICROBE_TARGET_MIN, min(MICROBE_TARGET_MAX, len(plant_names)))

    return Microbe(
        "Modified",
        rr,
        "beneficial",
        random.uniform(BENEFICIAL_EFFECT_MIN, BENEFICIAL_EFFECT_MAX),
        set(random.sample(plant_names, k)),
        MICROBE_INITIAL_COLONIES,
        True,
        MICROBE_INITIAL_COLONIES
    )


# =========================
# CORE
# =========================

def mismatch(rr, env):
    return any(not rr[v].contains(env[v].rating) for v in VARIABLES)


def plant_effects(plants, microbes):
    out = {p.name: {v: 0 for v in VARIABLES} for p in plants if p.alive}

    for p in plants:
        if not p.alive:
            continue
        ms = [m for m in microbes if m.alive and p.name in m.affects]
        if not ms:
            continue
        total = sum(m.colonies for m in ms)
        eff = sum(m.effect_size * m.colonies for m in ms) / total
        for v in VARIABLES:
            out[p.name][v] = eff
    return out


def suitability(rr, env):
    return sum((rr[v].midpoint - env[v].rating) ** 2 for v in VARIABLES)


def process_plants(env, plants, microbes):
    effects = plant_effects(plants, microbes)

    valid = []
    for p in plants:
        if not p.alive:
            continue

        eff_rr = {v: p.ranges[v].modified(effects[p.name][v]) for v in VARIABLES}

        if mismatch(eff_rr, env):
            p.alive = False
            p.specimens = 0
            continue

        valid.append((suitability(eff_rr, env), p))

    valid.sort(key=lambda x: x[0])

    for _, p in valid:
        need = p.specimens
        if all(env[v].amount >= need for v in FINITE_RESOURCE_BUCKETS):
            for v in FINITE_RESOURCE_BUCKETS:
                env[v].amount -= need
        else:
            p.alive = False
            p.specimens = 0

    total = sum(p.specimens for p in plants if p.alive)
    if total == 0:
        return

    for p in plants:
        if not p.alive:
            continue
        prob = (1 - p.specimens / total) * (1 - total / CARRYING_CAPACITY)
        p.specimens += sum(1 for _ in range(p.specimens) if random.random() < prob)


def process_microbes(env, plants, microbes):
    living = {p.name: p for p in plants if p.alive}

    for m in microbes:
        if not m.alive:
            continue

        if mismatch(m.effective_ranges(), env):
            m.alive = False
            m.colonies = 0
            continue

        targets = m.affects & living.keys()
        if not targets:
            m.alive = False
            m.colonies = 0
            continue

        cap = sum(living[t].specimens for t in targets)
        m.colonies = min(cap, m.colonies + 1)

def count_microbe_types(microbes):
    counts = {"beneficial": 0, "neutral": 0, "pathogenic": 0}

    for m in microbes:
        if m.alive and m.colonies > 0:
            counts[m.effect_type] += 1

    return counts

# =========================
# HGT (PER COLONY)
# =========================

def process_hgt(microbes):
    living = [m for m in microbes if m.alive and m.colonies > 0]

    total_colonies = sum(m.colonies for m in living)
    total_plasmid = sum(m.plasmid_colonies for m in living)

    if total_plasmid == 0:
        return 0

    percent = total_plasmid / total_colonies
    flat = random.uniform(HGT_FLAT_RATE_MIN, HGT_FLAT_RATE_MAX)

    transfers = 0

    for r in living:
        non_plasmid = r.colonies - r.plasmid_colonies
        if non_plasmid <= 0:
            continue

        for d in living:
            if d.plasmid_colonies == 0:
                continue

            shared = 2 if (r.affects & d.affects) else 1

            # number of donor–recipient interactions
            interactions = d.plasmid_colonies * non_plasmid

            p = clamp(flat * shared * percent, 0, 1)

            successes = 0
            for _ in range(interactions):
                if random.random() < p:
                    successes += 1

            # cannot exceed available non-plasmid colonies
            successes = min(successes, non_plasmid)

            if successes > 0:
                r.plasmid_colonies += successes
                non_plasmid -= successes
                transfers += successes

            if non_plasmid <= 0:
                break

    return transfers


# =========================
# REPORT
# =========================

def report(env, plants, microbes, label):
    print("=" * 100)
    print(label)
    print("=" * 100)

    print("\nBucket ratings:")
    for v, b in env.items():
        amt = "infinite" if math.isinf(b.amount) else f"{b.amount:.1f}"
        cap = "infinite" if math.isinf(b.capacity) else f"{b.capacity:.1f}"
        print(f"  {v}: rating={b.rating:.3f}, amount={amt}, capacity={cap}")

    alive_plants = [p for p in plants if p.alive and p.specimens > 0]
    alive_plants.sort(key=lambda x: -x.specimens)

    print(f"\nSurviving plant species: {len(alive_plants)}")
    for p in alive_plants:
        print(f"\n  {p.name}: specimens={p.specimens}")
        for v in VARIABLES:
            r = p.ranges[v]
            print(f"    {v}: [{r.low:.3f}, {r.high:.3f}] (mid={r.midpoint:.3f}, hr={r.half_range:.3f})")

    alive_microbes = [m for m in microbes if m.alive and m.colonies > 0]
    alive_microbes.sort(key=lambda x: -x.colonies)

    print(f"\nSurviving microbial strains: {len(alive_microbes)}")
    for m in alive_microbes:
        # Calculate plasmid_fraction here and then use it in the f-string
        plasmid_fraction = m.plasmid_colonies / m.colonies if m.colonies > 0 else 0
        print(
            f"\n  {m.name}: colonies={m.colonies}, "
            f"type={m.effect_type}, effect={m.effect_size:+.3f}, plasmid={plasmid_fraction:.2f}"
        )

def summary(plants_a, microbes_a, plants_b, microbes_b, hgt_total):
    a_plants = sum(1 for p in plants_a if p.alive and p.specimens > 0)
    a_microbes = sum(1 for m in microbes_a if m.alive and m.colonies > 0)

    b_plants = sum(1 for p in plants_b if p.alive and p.specimens > 0)
    b_microbes = sum(1 for m in microbes_b if m.alive and m.colonies > 0)

    a_counts = count_microbe_types(microbes_a)
    b_counts = count_microbe_types(microbes_b)

    print("=" * 100)
    print("SUMMARY")
    print("=" * 100)
    print(f"Environment A has {a_plants} plants and {a_microbes} microbes.")
    print(f"Environment B has {b_plants} plants and {b_microbes} microbes with {hgt_total} HGT events.")
    print(f"A microbe types: {a_counts}")
    print(f"B microbe types: {b_counts}")
    print()


# =========================
# SIM
# =========================

def run(seed=None):
    if seed is not None:
        random.seed(seed)

    env_a = make_environment()
    env_b = {k: Bucket(v.name, v.rating, v.capacity, v.amount, v.growth) for k, v in env_a.items()}

    plants_a = make_plants(N_PLANT_SPECIES)
    plants_b = [Plant(p.name, p.ranges, p.specimens) for p in plants_a]

    names = [p.name for p in plants_a]

    microbes_a = make_microbes(N_MICROBE_STRAINS, names)
    microbes_b = [
        Microbe(
            m.name,
            m.ranges,
            m.effect_type,
            m.effect_size,
            set(m.affects),
            m.colonies,
            m.alive,
            m.plasmid_colonies
        )
        for m in microbes_a
    ]

    if not CONTROL_MODE:
      microbes_b.append(make_modified_microbe(env_b, names))

    hgt_total = 0

    for t in range(1, N_TURNS + 1):
        if t >= 2:
            env_a["temperature"].rating = min(TEMPERATURE_MAX, env_a["temperature"].rating + TEMPERATURE_INCREMENT)
            env_b["temperature"].rating = min(TEMPERATURE_MAX, env_b["temperature"].rating + TEMPERATURE_INCREMENT)

        process_plants(env_a, plants_a, microbes_a)
        process_plants(env_b, plants_b, microbes_b)

        process_microbes(env_a, plants_a, microbes_a)
        process_microbes(env_b, plants_b, microbes_b)

        hgt_total += process_hgt(microbes_b)

        for b in env_a.values():
            b.replenish()
        for b in env_b.values():
            b.replenish()

    # SAVE TO CSV
    append_summary_to_csv(plants_a, microbes_a, plants_b, microbes_b, hgt_total, seed)

    summary(plants_a, microbes_a, plants_b, microbes_b, hgt_total)

    report(env_a, plants_a, microbes_a, "Environment_A")
    print()
    report(env_b, plants_b, microbes_b, "Environment_B")
    print("\nTotal HGT transfers:", hgt_total)


if __name__ == "__main__":
    for i in range(990):
        run()
from google.colab import files
files.download("run_summary_control.csv")

In [ ]:
import math
import random
import csv
import os
from dataclasses import dataclass
from typing import Dict, Set


# =========================
# CONFIG
# =========================

SEED = None

N_TURNS = 250
N_PLANT_SPECIES = 2000
N_MICROBE_STRAINS = 5000

VARIABLES = ["sunlight", "water", "carbon_dioxide", "nutrients", "temperature"]
FINITE_RESOURCE_BUCKETS = ["water", "carbon_dioxide", "nutrients"]

RESOURCE_CAPACITY = 50_000
RESOURCE_REPLENISH_FRACTION = 0.10
CARRYING_CAPACITY = 5_000

TEMPERATURE_START = 0.5
TEMPERATURE_MAX = 2.5
TEMPERATURE_INCREMENT = 0.001

PLANT_INITIAL_SPECIMENS = 20
MICROBE_INITIAL_COLONIES = 20

MICROBE_TYPE_WEIGHTS = {
    "neutral": 0.87,
    "beneficial": 0.12,
    "pathogenic": 0.01,
}

BENEFICIAL_EFFECT_MIN = 0.01
BENEFICIAL_EFFECT_MAX = 0.10
PATHOGENIC_EFFECT_MIN = -0.50
PATHOGENIC_EFFECT_MAX = -0.10

MICROBE_TARGET_MIN = 2
MICROBE_TARGET_MAX = 50

PLASMID_TEMP_BONUS = 0.15

HGT_FLAT_RATE_MIN = 0.001
HGT_FLAT_RATE_MAX = 0.1


# =========================
# HELPERS
# =========================

def clamp(x, lo, hi):
    return max(lo, min(hi, x))


def weighted_choice(d):
    r = random.random()
    s = 0
    for k, v in d.items():
        s += v
        if r <= s:
            return k
    return list(d.keys())[-1]


@dataclass
class Range:
    low: float
    high: float

    @property
    def midpoint(self):
        return (self.low + self.high) / 2

    @property
    def half_range(self):
        return (self.high - self.low) / 2

    def contains(self, x):
        return self.low <= x <= self.high

    def modified(self, effect):
        mid = self.midpoint
        hr = self.half_range * (1 + effect)
        return Range(
            clamp(mid - hr, 0.01, 1.0),
            clamp(mid + hr, 0.01, 1.0)
        )


@dataclass
class Bucket:
    name: str
    rating: float
    capacity: float
    amount: float
    growth: float

    def replenish(self):
        if self.name in FINITE_RESOURCE_BUCKETS:
            self.amount = min(self.capacity, self.amount + self.growth)


@dataclass
class Plant:
    name: str
    ranges: Dict[str, Range]
    specimens: int
    alive: bool = True


@dataclass
class Microbe:
    name: str
    ranges: Dict[str, Range]
    effect_type: str
    effect_size: float
    affects: Set[str]
    colonies: int
    alive: bool = True
    plasmid_colonies: int = 0

    def effective_ranges(self):
        rr = {k: Range(v.low, v.high) for k, v in self.ranges.items()}
        if self.colonies > 0 and self.plasmid_colonies > 0:
            t = rr["temperature"]
            rr["temperature"] = Range(t.low, clamp(t.high + PLASMID_TEMP_BONUS, 0.01, 1.0))
        return rr


# =========================
# CSV LOGGER
# =========================

def append_summary_to_csv(plants_a, microbes_a, plants_b, microbes_b, hgt_total, seed, filename="run_summary_different_ratios.csv"):
    a_plants = sum(1 for p in plants_a if p.alive and p.specimens > 0)
    a_microbes = sum(1 for m in microbes_a if m.alive and m.colonies > 0)

    b_plants = sum(1 for p in plants_b if p.alive and p.specimens > 0)
    b_microbes = sum(1 for m in microbes_b if m.alive and m.colonies > 0)

    file_exists = os.path.isfile(filename)

    with open(filename, "a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow([
                "seed",
                "A_plants", "A_microbes",
                "A_beneficial", "A_neutral", "A_pathogenic",
                "B_plants", "B_microbes",
                "B_beneficial", "B_neutral", "B_pathogenic",
                "HGT"
                ])
        a_counts = count_microbe_types(microbes_a)
        b_counts = count_microbe_types(microbes_b)

        writer.writerow([
            seed,
            a_plants, a_microbes,
            a_counts["beneficial"], a_counts["neutral"], a_counts["pathogenic"],
            b_plants, b_microbes,
            b_counts["beneficial"], b_counts["neutral"], b_counts["pathogenic"],
            hgt_total
        ])

# =========================
# GENERATION
# =========================

def random_range():
    return Range(random.uniform(0.01, 0.5), random.uniform(0.51, 1.0))


def make_environment():
    buckets = {}
    for v in VARIABLES:
        if v == "temperature":
            buckets[v] = Bucket(v, TEMPERATURE_START, float("inf"), float("inf"), 0)
        elif v == "sunlight":
            buckets[v] = Bucket(v, random.uniform(0.01, 1.0), float("inf"), float("inf"), 0)
        else:
            buckets[v] = Bucket(v, random.uniform(0.01, 1.0), RESOURCE_CAPACITY, RESOURCE_CAPACITY, RESOURCE_CAPACITY * 0.1)
    return buckets


def make_plants(n):
    return [Plant(f"P{i}", {v: random_range() for v in VARIABLES}, PLANT_INITIAL_SPECIMENS) for i in range(n)]


def make_microbes(n, plant_names):
    microbes = []
    for i in range(n):
        et = weighted_choice(MICROBE_TYPE_WEIGHTS)
        if et == "neutral":
            es = 0
        elif et == "beneficial":
            es = random.uniform(BENEFICIAL_EFFECT_MIN, BENEFICIAL_EFFECT_MAX)
        else:
            es = random.uniform(PATHOGENIC_EFFECT_MIN, PATHOGENIC_EFFECT_MAX)

        k = random.randint(MICROBE_TARGET_MIN, min(MICROBE_TARGET_MAX, len(plant_names)))

        microbes.append(
            Microbe(
                f"M{i}",
                {v: random_range() for v in VARIABLES},
                et,
                es,
                set(random.sample(plant_names, k)),
                MICROBE_INITIAL_COLONIES
            )
        )
    return microbes


def make_modified_microbe(env, plant_names):
    rr = {}
    for v in VARIABLES:
        val = env[v].rating
        rr[v] = Range(clamp(val - 0.25, 0.01, 1.0), clamp(val + 0.25, 0.01, 1.0))

    k = random.randint(MICROBE_TARGET_MIN, min(MICROBE_TARGET_MAX, len(plant_names)))

    return Microbe(
        "Modified",
        rr,
        "beneficial",
        random.uniform(BENEFICIAL_EFFECT_MIN, BENEFICIAL_EFFECT_MAX),
        set(random.sample(plant_names, k)),
        MICROBE_INITIAL_COLONIES,
        True,
        MICROBE_INITIAL_COLONIES
    )


# =========================
# CORE
# =========================

def mismatch(rr, env):
    return any(not rr[v].contains(env[v].rating) for v in VARIABLES)


def plant_effects(plants, microbes):
    out = {p.name: {v: 0 for v in VARIABLES} for p in plants if p.alive}

    for p in plants:
        if not p.alive:
            continue
        ms = [m for m in microbes if m.alive and p.name in m.affects]
        if not ms:
            continue
        total = sum(m.colonies for m in ms)
        eff = sum(m.effect_size * m.colonies for m in ms) / total
        for v in VARIABLES:
            out[p.name][v] = eff
    return out


def suitability(rr, env):
    return sum((rr[v].midpoint - env[v].rating) ** 2 for v in VARIABLES)


def process_plants(env, plants, microbes):
    effects = plant_effects(plants, microbes)

    valid = []
    for p in plants:
        if not p.alive:
            continue

        eff_rr = {v: p.ranges[v].modified(effects[p.name][v]) for v in VARIABLES}

        if mismatch(eff_rr, env):
            p.alive = False
            p.specimens = 0
            continue

        valid.append((suitability(eff_rr, env), p))

    valid.sort(key=lambda x: x[0])

    for _, p in valid:
        need = p.specimens
        if all(env[v].amount >= need for v in FINITE_RESOURCE_BUCKETS):
            for v in FINITE_RESOURCE_BUCKETS:
                env[v].amount -= need
        else:
            p.alive = False
            p.specimens = 0

    total = sum(p.specimens for p in plants if p.alive)
    if total == 0:
        return

    for p in plants:
        if not p.alive:
            continue
        prob = (1 - p.specimens / total) * (1 - total / CARRYING_CAPACITY)
        p.specimens += sum(1 for _ in range(p.specimens) if random.random() < prob)


def process_microbes(env, plants, microbes):
    living = {p.name: p for p in plants if p.alive}

    for m in microbes:
        if not m.alive:
            continue

        if mismatch(m.effective_ranges(), env):
            m.alive = False
            m.colonies = 0
            continue

        targets = m.affects & living.keys()
        if not targets:
            m.alive = False
            m.colonies = 0
            continue

        cap = sum(living[t].specimens for t in targets)
        m.colonies = min(cap, m.colonies + 1)

def count_microbe_types(microbes):
    counts = {"beneficial": 0, "neutral": 0, "pathogenic": 0}

    for m in microbes:
        if m.alive and m.colonies > 0:
            counts[m.effect_type] += 1

    return counts

# =========================
# HGT (PER COLONY)
# =========================

def process_hgt(microbes):
    living = [m for m in microbes if m.alive and m.colonies > 0]

    total_colonies = sum(m.colonies for m in living)
    total_plasmid = sum(m.plasmid_colonies for m in living)

    if total_plasmid == 0:
        return 0

    percent = total_plasmid / total_colonies
    flat = random.uniform(HGT_FLAT_RATE_MIN, HGT_FLAT_RATE_MAX)

    transfers = 0

    for r in living:
        non_plasmid = r.colonies - r.plasmid_colonies
        if non_plasmid <= 0:
            continue

        for d in living:
            if d.plasmid_colonies == 0:
                continue

            shared = 2 if (r.affects & d.affects) else 1

            # number of donor–recipient interactions
            interactions = d.plasmid_colonies * non_plasmid

            p = clamp(flat * shared * percent, 0, 1)

            successes = 0
            for _ in range(interactions):
                if random.random() < p:
                    successes += 1

            # cannot exceed available non-plasmid colonies
            successes = min(successes, non_plasmid)

            if successes > 0:
                r.plasmid_colonies += successes
                non_plasmid -= successes
                transfers += successes

            if non_plasmid <= 0:
                break

    return transfers


# =========================
# REPORT
# =========================

def report(env, plants, microbes, label):
    print("=" * 100)
    print(label)
    print("=" * 100)

    print("\nBucket ratings:")
    for v, b in env.items():
        amt = "infinite" if math.isinf(b.amount) else f"{b.amount:.1f}"
        cap = "infinite" if math.isinf(b.capacity) else f"{b.capacity:.1f}"
        print(f"  {v}: rating={b.rating:.3f}, amount={amt}, capacity={cap}")

    alive_plants = [p for p in plants if p.alive and p.specimens > 0]
    alive_plants.sort(key=lambda x: -x.specimens)

    print(f"\nSurviving plant species: {len(alive_plants)}")
    for p in alive_plants:
        print(f"\n  {p.name}: specimens={p.specimens}")
        for v in VARIABLES:
            r = p.ranges[v]
            print(f"    {v}: [{r.low:.3f}, {r.high:.3f}] (mid={r.midpoint:.3f}, hr={r.half_range:.3f})")

    alive_microbes = [m for m in microbes if m.alive and m.colonies > 0]
    alive_microbes.sort(key=lambda x: -x.colonies)

    print(f"\nSurviving microbial strains: {len(alive_microbes)}")
    for m in alive_microbes:
        # Calculate plasmid_fraction here and then use it in the f-string
        plasmid_fraction = m.plasmid_colonies / m.colonies if m.colonies > 0 else 0
        print(
            f"\n  {m.name}: colonies={m.colonies}, "
            f"type={m.effect_type}, effect={m.effect_size:+.3f}, plasmid={plasmid_fraction:.2f}"
        )

def summary(plants_a, microbes_a, plants_b, microbes_b, hgt_total):
    a_plants = sum(1 for p in plants_a if p.alive and p.specimens > 0)
    a_microbes = sum(1 for m in microbes_a if m.alive and m.colonies > 0)

    b_plants = sum(1 for p in plants_b if p.alive and p.specimens > 0)
    b_microbes = sum(1 for m in microbes_b if m.alive and m.colonies > 0)

    a_counts = count_microbe_types(microbes_a)
    b_counts = count_microbe_types(microbes_b)

    print("=" * 100)
    print("SUMMARY")
    print("=" * 100)
    print(f"Environment A has {a_plants} plants and {a_microbes} microbes.")
    print(f"Environment B has {b_plants} plants and {b_microbes} microbes with {hgt_total} HGT events.")
    print(f"A microbe types: {a_counts}")
    print(f"B microbe types: {b_counts}")
    print()


# =========================
# SIM
# =========================

def run(seed=None):
    if seed is not None:
        random.seed(seed)

    env_a = make_environment()
    env_b = {k: Bucket(v.name, v.rating, v.capacity, v.amount, v.growth) for k, v in env_a.items()}

    plants_a = make_plants(N_PLANT_SPECIES)
    plants_b = [Plant(p.name, p.ranges, p.specimens) for p in plants_a]

    names = [p.name for p in plants_a]

    microbes_a = make_microbes(N_MICROBE_STRAINS, names)
    microbes_b = [
        Microbe(
            m.name,
            m.ranges,
            m.effect_type,
            m.effect_size,
            set(m.affects),
            m.colonies,
            m.alive,
            m.plasmid_colonies
        )
        for m in microbes_a
    ]

    microbes_b.append(make_modified_microbe(env_b, names))

    hgt_total = 0

    for t in range(1, N_TURNS + 1):
        if t >= 2:
            env_a["temperature"].rating = min(TEMPERATURE_MAX, env_a["temperature"].rating + TEMPERATURE_INCREMENT)
            env_b["temperature"].rating = min(TEMPERATURE_MAX, env_b["temperature"].rating + TEMPERATURE_INCREMENT)

        process_plants(env_a, plants_a, microbes_a)
        process_plants(env_b, plants_b, microbes_b)

        process_microbes(env_a, plants_a, microbes_a)
        process_microbes(env_b, plants_b, microbes_b)

        hgt_total += process_hgt(microbes_b)

        for b in env_a.values():
            b.replenish()
        for b in env_b.values():
            b.replenish()

    # SAVE TO CSV
    append_summary_to_csv(plants_a, microbes_a, plants_b, microbes_b, hgt_total, seed)

    summary(plants_a, microbes_a, plants_b, microbes_b, hgt_total)

    report(env_a, plants_a, microbes_a, "Environment_A")
    print()
    report(env_b, plants_b, microbes_b, "Environment_B")
    print("\nTotal HGT transfers:", hgt_total)


if __name__ == "__main__":
    for i in range(700):
        run()
from google.colab import files
files.download("run_summary_different_ratios.csv")

In [ ]:
import math
import random
import csv
import os
from dataclasses import dataclass
from typing import Dict, Set


# =========================
# CONFIG
# =========================

SEED = None

N_TURNS = 350
TEMP_DELAY_TURNS = 100
N_PLANT_SPECIES = 2000
N_MICROBE_STRAINS = 5000

VARIABLES = ["sunlight", "water", "carbon_dioxide", "nutrients", "temperature"]
FINITE_RESOURCE_BUCKETS = ["water", "carbon_dioxide", "nutrients"]

RESOURCE_CAPACITY = 50_000
RESOURCE_REPLENISH_FRACTION = 0.10
CARRYING_CAPACITY = 5_000

TEMPERATURE_START = 0.5
TEMPERATURE_MAX = 2.5
TEMPERATURE_INCREMENT = 0.001

PLANT_INITIAL_SPECIMENS = 20
MICROBE_INITIAL_COLONIES = 20

MICROBE_TYPE_WEIGHTS = {
    "neutral": 0.85,
    "beneficial": 0.12,
    "pathogenic": 0.03,
}

BENEFICIAL_EFFECT_MIN = 0.01
BENEFICIAL_EFFECT_MAX = 0.10
PATHOGENIC_EFFECT_MIN = -0.50
PATHOGENIC_EFFECT_MAX = -0.10

MICROBE_TARGET_MIN = 2
MICROBE_TARGET_MAX = 50

PLASMID_TEMP_BONUS = 0.15

HGT_FLAT_RATE_MIN = 0.001
HGT_FLAT_RATE_MAX = 0.1


# =========================
# HELPERS
# =========================

def clamp(x, lo, hi):
    return max(lo, min(hi, x))


def weighted_choice(d):
    r = random.random()
    s = 0
    for k, v in d.items():
        s += v
        if r <= s:
            return k
    return list(d.keys())[-1]


@dataclass
class Range:
    low: float
    high: float

    @property
    def midpoint(self):
        return (self.low + self.high) / 2

    @property
    def half_range(self):
        return (self.high - self.low) / 2

    def contains(self, x):
        return self.low <= x <= self.high

    def modified(self, effect):
        mid = self.midpoint
        hr = self.half_range * (1 + effect)
        return Range(
            clamp(mid - hr, 0.01, 1.0),
            clamp(mid + hr, 0.01, 1.0)
        )


@dataclass
class Bucket:
    name: str
    rating: float
    capacity: float
    amount: float
    growth: float

    def replenish(self):
        if self.name in FINITE_RESOURCE_BUCKETS:
            self.amount = min(self.capacity, self.amount + self.growth)


@dataclass
class Plant:
    name: str
    ranges: Dict[str, Range]
    specimens: int
    alive: bool = True


@dataclass
class Microbe:
    name: str
    ranges: Dict[str, Range]
    effect_type: str
    effect_size: float
    affects: Set[str]
    colonies: int
    alive: bool = True
    plasmid_colonies: int = 0

    def effective_ranges(self):
        rr = {k: Range(v.low, v.high) for k, v in self.ranges.items()}
        if self.colonies > 0 and self.plasmid_colonies > 0:
            t = rr["temperature"]
            rr["temperature"] = Range(t.low, clamp(t.high + PLASMID_TEMP_BONUS, 0.01, 1.0))
        return rr


# =========================
# CSV LOGGER
# =========================

def append_summary_to_csv(plants_a, microbes_a, plants_b, microbes_b, hgt_total, seed, filename="run_summary.csv"):
    a_plants = sum(1 for p in plants_a if p.alive and p.specimens > 0)
    a_microbes = sum(1 for m in microbes_a if m.alive and m.colonies > 0)

    b_plants = sum(1 for p in plants_b if p.alive and p.specimens > 0)
    b_microbes = sum(1 for m in microbes_b if m.alive and m.colonies > 0)

    file_exists = os.path.isfile(filename)

    with open(filename, "a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow([
                "seed",
                "A_plants", "A_microbes",
                "A_beneficial", "A_neutral", "A_pathogenic",
                "B_plants", "B_microbes",
                "B_beneficial", "B_neutral", "B_pathogenic",
                "HGT"
                ])
        a_counts = count_microbe_types(microbes_a)
        b_counts = count_microbe_types(microbes_b)

        writer.writerow([
            seed,
            a_plants, a_microbes,
            a_counts["beneficial"], a_counts["neutral"], a_counts["pathogenic"],
            b_plants, b_microbes,
            b_counts["beneficial"], b_counts["neutral"], b_counts["pathogenic"],
            hgt_total
        ])

# =========================
# GENERATION
# =========================

def random_range():
    return Range(random.uniform(0.01, 0.5), random.uniform(0.51, 1.0))


def make_environment():
    buckets = {}
    for v in VARIABLES:
        if v == "temperature":
            buckets[v] = Bucket(v, TEMPERATURE_START, float("inf"), float("inf"), 0)
        elif v == "sunlight":
            buckets[v] = Bucket(v, random.uniform(0.01, 1.0), float("inf"), float("inf"), 0)
        else:
            buckets[v] = Bucket(v, random.uniform(0.01, 1.0), RESOURCE_CAPACITY, RESOURCE_CAPACITY, RESOURCE_CAPACITY * 0.1)
    return buckets


def make_plants(n):
    return [Plant(f"P{i}", {v: random_range() for v in VARIABLES}, PLANT_INITIAL_SPECIMENS) for i in range(n)]


def make_microbes(n, plant_names):
    microbes = []
    for i in range(n):
        et = weighted_choice(MICROBE_TYPE_WEIGHTS)
        if et == "neutral":
            es = 0
        elif et == "beneficial":
            es = random.uniform(BENEFICIAL_EFFECT_MIN, BENEFICIAL_EFFECT_MAX)
        else:
            es = random.uniform(PATHOGENIC_EFFECT_MIN, PATHOGENIC_EFFECT_MAX)

        k = random.randint(MICROBE_TARGET_MIN, min(MICROBE_TARGET_MAX, len(plant_names)))

        microbes.append(
            Microbe(
                f"M{i}",
                {v: random_range() for v in VARIABLES},
                et,
                es,
                set(random.sample(plant_names, k)),
                MICROBE_INITIAL_COLONIES
            )
        )
    return microbes


def make_modified_microbe(env, plant_names):
    rr = {}
    for v in VARIABLES:
        val = env[v].rating
        rr[v] = Range(clamp(val - 0.25, 0.01, 1.0), clamp(val + 0.25, 0.01, 1.0))

    k = random.randint(MICROBE_TARGET_MIN, min(MICROBE_TARGET_MAX, len(plant_names)))

    return Microbe(
        "Modified",
        rr,
        "beneficial",
        random.uniform(BENEFICIAL_EFFECT_MIN, BENEFICIAL_EFFECT_MAX),
        set(random.sample(plant_names, k)),
        MICROBE_INITIAL_COLONIES,
        True,
        MICROBE_INITIAL_COLONIES
    )


# =========================
# CORE
# =========================

def mismatch(rr, env):
    return any(not rr[v].contains(env[v].rating) for v in VARIABLES)


def plant_effects(plants, microbes):
    out = {p.name: {v: 0 for v in VARIABLES} for p in plants if p.alive}

    for p in plants:
        if not p.alive:
            continue
        ms = [m for m in microbes if m.alive and p.name in m.affects]
        if not ms:
            continue
        total = sum(m.colonies for m in ms)
        eff = sum(m.effect_size * m.colonies for m in ms) / total
        for v in VARIABLES:
            out[p.name][v] = eff
    return out


def suitability(rr, env):
    return sum((rr[v].midpoint - env[v].rating) ** 2 for v in VARIABLES)


def process_plants(env, plants, microbes):
    effects = plant_effects(plants, microbes)

    valid = []
    for p in plants:
        if not p.alive:
            continue

        eff_rr = {v: p.ranges[v].modified(effects[p.name][v]) for v in VARIABLES}

        if mismatch(eff_rr, env):
            p.alive = False
            p.specimens = 0
            continue

        valid.append((suitability(eff_rr, env), p))

    valid.sort(key=lambda x: x[0])

    for _, p in valid:
        need = p.specimens
        if all(env[v].amount >= need for v in FINITE_RESOURCE_BUCKETS):
            for v in FINITE_RESOURCE_BUCKETS:
                env[v].amount -= need
        else:
            p.alive = False
            p.specimens = 0

    total = sum(p.specimens for p in plants if p.alive)
    if total == 0:
        return

    for p in plants:
        if not p.alive:
            continue
        prob = (1 - p.specimens / total) * (1 - total / CARRYING_CAPACITY)
        p.specimens += sum(1 for _ in range(p.specimens) if random.random() < prob)


def process_microbes(env, plants, microbes):
    living = {p.name: p for p in plants if p.alive}

    for m in microbes:
        if not m.alive:
            continue

        if mismatch(m.effective_ranges(), env):
            m.alive = False
            m.colonies = 0
            continue

        targets = m.affects & living.keys()
        if not targets:
            m.alive = False
            m.colonies = 0
            continue

        cap = sum(living[t].specimens for t in targets)
        m.colonies = min(cap, m.colonies + 1)

def count_microbe_types(microbes):
    counts = {"beneficial": 0, "neutral": 0, "pathogenic": 0}

    for m in microbes:
        if m.alive and m.colonies > 0:
            counts[m.effect_type] += 1

    return counts

# =========================
# HGT (PER COLONY)
# =========================

def process_hgt(microbes):
    living = [m for m in microbes if m.alive and m.colonies > 0]

    total_colonies = sum(m.colonies for m in living)
    total_plasmid = sum(m.plasmid_colonies for m in living)

    if total_plasmid == 0:
        return 0

    percent = total_plasmid / total_colonies
    flat = random.uniform(HGT_FLAT_RATE_MIN, HGT_FLAT_RATE_MAX)

    transfers = 0

    for r in living:
        non_plasmid = r.colonies - r.plasmid_colonies
        if non_plasmid <= 0:
            continue

        for d in living:
            if d.plasmid_colonies == 0:
                continue

            shared = 2 if (r.affects & d.affects) else 1

            # number of donor–recipient interactions
            interactions = d.plasmid_colonies * non_plasmid

            p = clamp(flat * shared * percent, 0, 1)

            successes = 0
            for _ in range(interactions):
                if random.random() < p:
                    successes += 1

            # cannot exceed available non-plasmid colonies
            successes = min(successes, non_plasmid)

            if successes > 0:
                r.plasmid_colonies += successes
                non_plasmid -= successes
                transfers += successes

            if non_plasmid <= 0:
                break

    return transfers


# =========================
# REPORT
# =========================

def report(env, plants, microbes, label):
    print("=" * 100)
    print(label)
    print("=" * 100)

    print("\nBucket ratings:")
    for v, b in env.items():
        amt = "infinite" if math.isinf(b.amount) else f"{b.amount:.1f}"
        cap = "infinite" if math.isinf(b.capacity) else f"{b.capacity:.1f}"
        print(f"  {v}: rating={b.rating:.3f}, amount={amt}, capacity={cap}")

    alive_plants = [p for p in plants if p.alive and p.specimens > 0]
    alive_plants.sort(key=lambda x: -x.specimens)

    print(f"\nSurviving plant species: {len(alive_plants)}")
    for p in alive_plants:
        print(f"\n  {p.name}: specimens={p.specimens}")
        for v in VARIABLES:
            r = p.ranges[v]
            print(f"    {v}: [{r.low:.3f}, {r.high:.3f}] (mid={r.midpoint:.3f}, hr={r.half_range:.3f})")

    alive_microbes = [m for m in microbes if m.alive and m.colonies > 0]
    alive_microbes.sort(key=lambda x: -x.colonies)

    print(f"\nSurviving microbial strains: {len(alive_microbes)}")
    for m in alive_microbes:
        # Calculate plasmid_fraction here and then use it in the f-string
        plasmid_fraction = m.plasmid_colonies / m.colonies if m.colonies > 0 else 0
        print(
            f"\n  {m.name}: colonies={m.colonies}, "
            f"type={m.effect_type}, effect={m.effect_size:+.3f}, plasmid={plasmid_fraction:.2f}"
        )

def summary(plants_a, microbes_a, plants_b, microbes_b, hgt_total):
    a_plants = sum(1 for p in plants_a if p.alive and p.specimens > 0)
    a_microbes = sum(1 for m in microbes_a if m.alive and m.colonies > 0)

    b_plants = sum(1 for p in plants_b if p.alive and p.specimens > 0)
    b_microbes = sum(1 for m in microbes_b if m.alive and m.colonies > 0)

    a_counts = count_microbe_types(microbes_a)
    b_counts = count_microbe_types(microbes_b)

    print("=" * 100)
    print("SUMMARY")
    print("=" * 100)
    print(f"Environment A has {a_plants} plants and {a_microbes} microbes.")
    print(f"Environment B has {b_plants} plants and {b_microbes} microbes with {hgt_total} HGT events.")
    print(f"A microbe types: {a_counts}")
    print(f"B microbe types: {b_counts}")
    print()


# =========================
# SIM
# =========================
def run(seed=None):
    if seed is not None:
        random.seed(seed)

    env_a = make_environment()
    env_b = {k: Bucket(v.name, v.rating, v.capacity, v.amount, v.growth) for k, v in env_a.items()}

    plants_a = make_plants(N_PLANT_SPECIES)
    plants_b = [Plant(p.name, p.ranges, p.specimens) for p in plants_a]

    names = [p.name for p in plants_a]

    microbes_a = make_microbes(N_MICROBE_STRAINS, names)
    microbes_b = [
        Microbe(
            m.name,
            m.ranges,
            m.effect_type,
            m.effect_size,
            set(m.affects),
            m.colonies,
            m.alive,
            m.plasmid_colonies
        )
        for m in microbes_a
    ]

    hgt_total = 0
    modified_microbe_added = False

    for t in range(1, N_TURNS + 1):
        if t > TEMP_DELAY_TURNS:
            env_a["temperature"].rating = min(
                TEMPERATURE_MAX,
                env_a["temperature"].rating + TEMPERATURE_INCREMENT
            )
            env_b["temperature"].rating = min(
                TEMPERATURE_MAX,
                env_b["temperature"].rating + TEMPERATURE_INCREMENT
            )

            if not modified_microbe_added:
                microbes_b.append(make_modified_microbe(env_b, names))
                modified_microbe_added = True

        process_plants(env_a, plants_a, microbes_a)
        process_plants(env_b, plants_b, microbes_b)

        process_microbes(env_a, plants_a, microbes_a)
        process_microbes(env_b, plants_b, microbes_b)

        hgt_total += process_hgt(microbes_b)

        for b in env_a.values():
            b.replenish()
        for b in env_b.values():
            b.replenish()

    append_summary_to_csv(plants_a, microbes_a, plants_b, microbes_b, hgt_total, seed)

    summary(plants_a, microbes_a, plants_b, microbes_b, hgt_total)

    report(env_a, plants_a, microbes_a, "Environment_A")
    print()
    report(env_b, plants_b, microbes_b, "Environment_B")
    print("\nTotal HGT transfers:", hgt_total)


if __name__ == "__main__":
    for i in range(990):
        run()
from google.colab import files
files.download("run_summary.csv")

In [ ]:
import math
import random
import csv
import os
from dataclasses import dataclass
from typing import Dict, Set


# =========================
# CONFIG
# =========================

SEED = None

N_TURNS = 250
N_PLANT_SPECIES = 2000
N_MICROBE_STRAINS = 5000

VARIABLES = ["sunlight", "water", "carbon_dioxide", "nutrients", "temperature"]
FINITE_RESOURCE_BUCKETS = ["water", "carbon_dioxide", "nutrients"]

RESOURCE_CAPACITY = 50_000
RESOURCE_REPLENISH_FRACTION = 0.10
CARRYING_CAPACITY = 5_000

TEMPERATURE_START = 0.5
TEMPERATURE_MAX = 2.5
TEMPERATURE_INCREMENT = 0.001

PLANT_INITIAL_SPECIMENS = 20
MICROBE_INITIAL_COLONIES = 20

MICROBE_TYPE_WEIGHTS = {
    "neutral": 0.85,
    "beneficial": 0.12,
    "pathogenic": 0.03,
}

BENEFICIAL_EFFECT_MIN = 0.01
BENEFICIAL_EFFECT_MAX = 0.10
PATHOGENIC_EFFECT_MIN = -0.50
PATHOGENIC_EFFECT_MAX = -0.10

MICROBE_TARGET_MIN = 2
MICROBE_TARGET_MAX = 50

PLASMID_TEMP_BONUS = 0.15

HGT_FLAT_RATE_MIN = 0.001
HGT_FLAT_RATE_MAX = 0.1


# =========================
# HELPERS
# =========================

def clamp(x, lo, hi):
    return max(lo, min(hi, x))


def weighted_choice(d):
    r = random.random()
    s = 0
    for k, v in d.items():
        s += v
        if r <= s:
            return k
    return list(d.keys())[-1]


@dataclass
class Range:
    low: float
    high: float

    @property
    def midpoint(self):
        return (self.low + self.high) / 2

    @property
    def half_range(self):
        return (self.high - self.low) / 2

    def contains(self, x):
        return self.low <= x <= self.high

    def modified(self, effect):
        mid = self.midpoint
        hr = self.half_range * (1 + effect)
        return Range(
            clamp(mid - hr, 0.01, 1.0),
            clamp(mid + hr, 0.01, 1.0)
        )


@dataclass
class Bucket:
    name: str
    rating: float
    capacity: float
    amount: float
    growth: float

    def replenish(self):
        if self.name in FINITE_RESOURCE_BUCKETS:
            self.amount = min(self.capacity, self.amount + self.growth)


@dataclass
class Plant:
    name: str
    ranges: Dict[str, Range]
    specimens: int
    alive: bool = True


@dataclass
class Microbe:
    name: str
    ranges: Dict[str, Range]
    effect_type: str
    effect_size: float
    affects: Set[str]
    colonies: int
    alive: bool = True
    plasmid_colonies: int = 0

    def effective_ranges(self):
        rr = {k: Range(v.low, v.high) for k, v in self.ranges.items()}
        if self.colonies > 0 and self.plasmid_colonies > 0:
            t = rr["temperature"]
            rr["temperature"] = Range(t.low, clamp(t.high + PLASMID_TEMP_BONUS, 0.01, 1.0))
        return rr


# =========================
# CSV LOGGER
# =========================

def append_summary_to_csv(plants_a, microbes_a, plants_b, microbes_b, hgt_total, seed, filename="run_summary_different_ratios.csv"):
    a_plants = sum(1 for p in plants_a if p.alive and p.specimens > 0)
    a_microbes = sum(1 for m in microbes_a if m.alive and m.colonies > 0)

    b_plants = sum(1 for p in plants_b if p.alive and p.specimens > 0)
    b_microbes = sum(1 for m in microbes_b if m.alive and m.colonies > 0)

    file_exists = os.path.isfile(filename)

    with open(filename, "a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow([
                "seed",
                "A_plants", "A_microbes",
                "A_beneficial", "A_neutral", "A_pathogenic",
                "B_plants", "B_microbes",
                "B_beneficial", "B_neutral", "B_pathogenic",
                "HGT"
                ])
        a_counts = count_microbe_types(microbes_a)
        b_counts = count_microbe_types(microbes_b)

        writer.writerow([
            seed,
            a_plants, a_microbes,
            a_counts["beneficial"], a_counts["neutral"], a_counts["pathogenic"],
            b_plants, b_microbes,
            b_counts["beneficial"], b_counts["neutral"], b_counts["pathogenic"],
            hgt_total
        ])

# =========================
# GENERATION
# =========================

def random_range():
    return Range(random.uniform(0.01, 0.5), random.uniform(0.51, 1.0))


def make_environment():
    buckets = {}
    for v in VARIABLES:
        if v == "temperature":
            buckets[v] = Bucket(v, TEMPERATURE_START, float("inf"), float("inf"), 0)
        elif v == "sunlight":
            buckets[v] = Bucket(v, random.uniform(0.01, 1.0), float("inf"), float("inf"), 0)
        else:
            buckets[v] = Bucket(v, random.uniform(0.01, 1.0), RESOURCE_CAPACITY, RESOURCE_CAPACITY, RESOURCE_CAPACITY * 0.1)
    return buckets


def make_plants(n):
    return [Plant(f"P{i}", {v: random_range() for v in VARIABLES}, PLANT_INITIAL_SPECIMENS) for i in range(n)]


def make_microbes(n, plant_names):
    microbes = []
    for i in range(n):
        et = weighted_choice(MICROBE_TYPE_WEIGHTS)
        if et == "neutral":
            es = 0
        elif et == "beneficial":
            es = random.uniform(BENEFICIAL_EFFECT_MIN, BENEFICIAL_EFFECT_MAX)
        else:
            es = random.uniform(PATHOGENIC_EFFECT_MIN, PATHOGENIC_EFFECT_MAX)

        k = random.randint(MICROBE_TARGET_MIN, min(MICROBE_TARGET_MAX, len(plant_names)))

        microbes.append(
            Microbe(
                f"M{i}",
                {v: random_range() for v in VARIABLES},
                et,
                es,
                set(random.sample(plant_names, k)),
                MICROBE_INITIAL_COLONIES
            )
        )
    return microbes


def make_modified_microbe(env, plant_names):
    rr = {}
    for v in VARIABLES:
        val = env[v].rating
        rr[v] = Range(clamp(val - 0.25, 0.01, 1.0), clamp(val + 0.25, 0.01, 1.0))

    k = random.randint(MICROBE_TARGET_MIN, min(MICROBE_TARGET_MAX, len(plant_names)))

    return Microbe(
        "Modified",
        rr,
        "beneficial",
        random.uniform(BENEFICIAL_EFFECT_MIN, BENEFICIAL_EFFECT_MAX),
        set(random.sample(plant_names, k)),
        MICROBE_INITIAL_COLONIES,
        True,
        MICROBE_INITIAL_COLONIES
    )


# =========================
# CORE
# =========================

def mismatch(rr, env):
    return any(not rr[v].contains(env[v].rating) for v in VARIABLES)


def plant_effects(plants, microbes):
    out = {p.name: {v: 0 for v in VARIABLES} for p in plants if p.alive}

    for p in plants:
        if not p.alive:
            continue
        ms = [m for m in microbes if m.alive and p.name in m.affects]
        if not ms:
            continue
        total = sum(m.colonies for m in ms)
        eff = sum(m.effect_size * m.colonies for m in ms) / total
        for v in VARIABLES:
            out[p.name][v] = eff
    return out


def suitability(rr, env):
    return sum((rr[v].midpoint - env[v].rating) ** 2 for v in VARIABLES)


def process_plants(env, plants, microbes):
    effects = plant_effects(plants, microbes)

    valid = []
    for p in plants:
        if not p.alive:
            continue

        eff_rr = {v: p.ranges[v].modified(effects[p.name][v]) for v in VARIABLES}

        if mismatch(eff_rr, env):
            p.alive = False
            p.specimens = 0
            continue

        valid.append((suitability(eff_rr, env), p))

    valid.sort(key=lambda x: x[0])

    for _, p in valid:
        need = p.specimens
        if all(env[v].amount >= need for v in FINITE_RESOURCE_BUCKETS):
            for v in FINITE_RESOURCE_BUCKETS:
                env[v].amount -= need
        else:
            p.alive = False
            p.specimens = 0

    total = sum(p.specimens for p in plants if p.alive)
    if total == 0:
        return

    for p in plants:
        if not p.alive:
            continue
        prob = (1 - p.specimens / total) * (1 - total / CARRYING_CAPACITY)
        p.specimens += sum(1 for _ in range(p.specimens) if random.random() < prob)


def process_microbes(env, plants, microbes):
    living = {p.name: p for p in plants if p.alive}

    for m in microbes:
        if not m.alive:
            continue

        if mismatch(m.effective_ranges(), env):
            m.alive = False
            m.colonies = 0
            continue

        targets = m.affects & living.keys()
        if not targets:
            m.alive = False
            m.colonies = 0
            continue

        cap = sum(living[t].specimens for t in targets)
        m.colonies = min(cap, m.colonies + 1)

def count_microbe_types(microbes):
    counts = {"beneficial": 0, "neutral": 0, "pathogenic": 0}

    for m in microbes:
        if m.alive and m.colonies > 0:
            counts[m.effect_type] += 1

    return counts

# =========================
# HGT (PER COLONY)
# =========================

def process_hgt(microbes):
    living = [m for m in microbes if m.alive and m.colonies > 0]

    total_colonies = sum(m.colonies for m in living)
    total_plasmid = sum(m.plasmid_colonies for m in living)

    if total_plasmid == 0:
        return 0

    percent = total_plasmid / total_colonies
    flat = random.uniform(HGT_FLAT_RATE_MIN, HGT_FLAT_RATE_MAX)

    transfers = 0

    for r in living:
        non_plasmid = r.colonies - r.plasmid_colonies
        if non_plasmid <= 0:
            continue

        for d in living:
            if d.plasmid_colonies == 0:
                continue

            shared = 2 if (r.affects & d.affects) else 1

            # number of donor–recipient interactions
            interactions = d.plasmid_colonies * non_plasmid

            p = clamp(flat * shared * percent, 0, 1)

            successes = 0
            for _ in range(interactions):
                if random.random() < p:
                    successes += 1

            # cannot exceed available non-plasmid colonies
            successes = min(successes, non_plasmid)

            if successes > 0:
                r.plasmid_colonies += successes
                non_plasmid -= successes
                transfers += successes

            if non_plasmid <= 0:
                break

    return transfers


# =========================
# REPORT
# =========================

def report(env, plants, microbes, label):
    print("=" * 100)
    print(label)
    print("=" * 100)

    print("\nBucket ratings:")
    for v, b in env.items():
        amt = "infinite" if math.isinf(b.amount) else f"{b.amount:.1f}"
        cap = "infinite" if math.isinf(b.capacity) else f"{b.capacity:.1f}"
        print(f"  {v}: rating={b.rating:.3f}, amount={amt}, capacity={cap}")

    alive_plants = [p for p in plants if p.alive and p.specimens > 0]
    alive_plants.sort(key=lambda x: -x.specimens)

    print(f"\nSurviving plant species: {len(alive_plants)}")
    for p in alive_plants:
        print(f"\n  {p.name}: specimens={p.specimens}")
        for v in VARIABLES:
            r = p.ranges[v]
            print(f"    {v}: [{r.low:.3f}, {r.high:.3f}] (mid={r.midpoint:.3f}, hr={r.half_range:.3f})")

    alive_microbes = [m for m in microbes if m.alive and m.colonies > 0]
    alive_microbes.sort(key=lambda x: -x.colonies)

    print(f"\nSurviving microbial strains: {len(alive_microbes)}")
    for m in alive_microbes:
        # Calculate plasmid_fraction here and then use it in the f-string
        plasmid_fraction = m.plasmid_colonies / m.colonies if m.colonies > 0 else 0
        print(
            f"\n  {m.name}: colonies={m.colonies}, "
            f"type={m.effect_type}, effect={m.effect_size:+.3f}, plasmid={plasmid_fraction:.2f}"
        )

def summary(plants_a, microbes_a, plants_b, microbes_b, hgt_total):
    a_plants = sum(1 for p in plants_a if p.alive and p.specimens > 0)
    a_microbes = sum(1 for m in microbes_a if m.alive and m.colonies > 0)

    b_plants = sum(1 for p in plants_b if p.alive and p.specimens > 0)
    b_microbes = sum(1 for m in microbes_b if m.alive and m.colonies > 0)

    a_counts = count_microbe_types(microbes_a)
    b_counts = count_microbe_types(microbes_b)

    print("=" * 100)
    print("SUMMARY")
    print("=" * 100)
    print(f"Environment A has {a_plants} plants and {a_microbes} microbes.")
    print(f"Environment B has {b_plants} plants and {b_microbes} microbes with {hgt_total} HGT events.")
    print(f"A microbe types: {a_counts}")
    print(f"B microbe types: {b_counts}")
    print()


# =========================
# SIM
# =========================

def run(seed=None):
    if seed is not None:
        random.seed(seed)

    env_a = make_environment()
    env_b = {k: Bucket(v.name, v.rating, v.capacity, v.amount, v.growth) for k, v in env_a.items()}

    plants_a = make_plants(N_PLANT_SPECIES)
    plants_b = [Plant(p.name, p.ranges, p.specimens) for p in plants_a]

    names = [p.name for p in plants_a]

    microbes_a = make_microbes(N_MICROBE_STRAINS, names)
    microbes_b = [
        Microbe(
            m.name,
            m.ranges,
            m.effect_type,
            m.effect_size,
            set(m.affects),
            m.colonies,
            m.alive,
            m.plasmid_colonies
        )
        for m in microbes_a
    ]

    microbes_b.append(make_modified_microbe(env_b, names))

    hgt_total = 0

    for t in range(1, N_TURNS + 1):
        if t >= 2:
            env_a["temperature"].rating = min(TEMPERATURE_MAX, env_a["temperature"].rating + TEMPERATURE_INCREMENT)
            env_b["temperature"].rating = min(TEMPERATURE_MAX, env_b["temperature"].rating + TEMPERATURE_INCREMENT)

        process_plants(env_a, plants_a, microbes_a)
        process_plants(env_b, plants_b, microbes_b)

        process_microbes(env_a, plants_a, microbes_a)
        process_microbes(env_b, plants_b, microbes_b)

        hgt_total += process_hgt(microbes_b)

        for b in env_a.values():
            b.replenish()
        for b in env_b.values():
            b.replenish()

    # SAVE TO CSV
    append_summary_to_csv(plants_a, microbes_a, plants_b, microbes_b, hgt_total, seed)

    summary(plants_a, microbes_a, plants_b, microbes_b, hgt_total)

    report(env_a, plants_a, microbes_a, "Environment_A")
    print()
    report(env_b, plants_b, microbes_b, "Environment_B")
    print("\nTotal HGT transfers:", hgt_total)


if __name__ == "__main__":
    for i in range(700):
        run()
from google.colab import files
files.download("run_summary_different_ratios.csv")